# SWE-HorizonBench Paper Reproduction Notebook

Executable companion for *Why Smoke Gains Fail on the Host: Soundness, Punica Throughput, and the 150-Line Horizon in 4-Bit SWE Agents*  
**IP-sanitized** · **no agent zips** · **no holdout IDs** · runs offline on the companion dataset (`moranzavdi/gemma4-soundness-and-punica-bench`). Every table below is recomputed from the shipped JSON/JSONL/CSV.


# Why Smoke Gains Fail on the Host: Soundness, Punica Throughput, and the 150-Line Horizon in 4-Bit SWE Agents

**Authors:** Moran Zavdi (team)  
**Host model:** `gemma-4-31b-it-qat-w4a16-ct` (QAT W4A16)  
**Resources:** Kaggle dataset `moranzavdi/gemma4-soundness-and-punica-bench` (data, `swe_horizon_bench/`, executed notebook).

---

## Abstract

**Problem.** Under a 12 h limit, local smoke gains often vanish on the leaderboard, and broken harnesses distort Pass@1. **Methodological insight.** Under tight budgets, agent performance is set jointly by **test soundness**, **serving speed**, **context retrieval**, and **first-shot patching**—not model weights alone: rank-64 LoRA breaks fused 4-bit execution, and a 150-line read cap hides companion edits. **Strongest result.** Two-sided gating keeps **111/129** tasks; on an $N=5$ timing core, LoRA matches base Pass@1 (**0.40**) but cuts episode throughput **20.1→11.0 tok/s** (+87 s/task; **+90.5 s** on $N=15$, $p<0.001$); across 85 usable tasks the agent reads a gold file on **84%** of runs yet solves **16%**, with second-edit repair at **0/10**. **Limitations & practical implication.** Paired E2E samples are small ($N=5$ timed; 15 censored runs over 10 tasks) and the OOD probe is synthetic; within those bounds, the best recipe is base inference (`timeout_seconds=240`, **0.18** LB) plus cost-aware rollout selection (**MS-SFT**). Nine retrieval baselines show our graph reader's gain is lexical; file finding is the bottleneck.

---

## 1. Introduction

Under a hard clock, improving a coding agent's weights is not enough: its score is set jointly by **test soundness** (§3), **serving speed** (§§4–5.2), **context retrieval** (§5.3, §6.1), and **first-shot patching** (§5.3, §6.2). On `gemma-4-31b-it-qat-w4a16-ct` under a ∼12 h limit (∼0.017/task), matched base-vs-LoRA runs hit identical Pass@1 (**0.40**, $N=5$) while LoRA adds **+87.2 s**/task (**+90.5 s** across **$N=15$** pairs, $p<0.001$) at 33% lower episode throughput (§5.2). The drop from reading a gold file to passing (**84%→16%**), the 150-line read cap, and **0/10** second-edit repair motivate our retrieval and rollout-selection experiments (§6). Three parts follow: **Part I (Diagnosis, §§3–5)**, **Part II (Methodology, §6)**, **Part III (Validation, §§7–10)**.

---

## 2. Methods

**Protocol.** The agent calls `run_command`, `read_file`, `edit_file`, `write_file`, `get_status`, and `submit_patch` (+ optional graph tools) at `temp=0.2`, `top_p=0.95`, `out=8192`, `think=4096` under 7–8 min / 28-call caps and `timeout_seconds`∈{240,300}.

**`submit_patch` trap.** ADK runs `git add -N .` before diffing, sweeping untracked scratch into the patch; the verifier resets protected tests, **not** scratch.

**Budgets vs grading.** `max_tool_calls` caps four tools; call exhaustion still grades a valid working-tree diff.

**Statistics.** Non-holdout only; 10,000-bootstrap 95% CIs and exact Wilcoxon or sign-flip $p$.

---

## 3. Part I: Diagnosis — Soundness Census

*Why do software engineering agents fail under constrained deployment?*

### Table 1 — Empty vs oracle on 129 tasks

| Bucket | $n$ | Rule |
| --- | ---: | --- |
| Usable | **111** | empty fails ∧ oracle passes |
| Unusable | **18** | empty-green, oracle-red, or both-green |
| Usable non-holdout | **85** | published per-task ledger |

### Table 1b — Pin recovery (compact)

| Intervention | Effect |
| --- | --- |
| Starlette **0.46.2** + `PYTHONPATH=src` | Unlocks **5** quarantined tasks (**18→13** unusable) |
| Python **3.12** + pytest `--timeout=5` | Recovers **11/11** native-verifier skips |
| Python **3.14** empty-green false solves | Catches `fastapi_14246`, `fastapi_14609` |

![Figure 1](figures/fig1_soundness_and_drift.svg)

---

In [4]:

from pathlib import Path
import json, pandas as pd, matplotlib.pyplot as plt
from IPython.display import SVG, display

ROOT = Path('..') if Path('../data').exists() else Path('.')
# Companion dataset: Kaggle mount, notebook-adjacent data/, or the repo mirror
CANDS = [
    Path('/kaggle/input/gemma4-soundness-and-punica-bench'),
    Path('/kaggle/input/swe-horizon-bench'),
    ROOT / 'data',
    Path('../../gemma4-agent/results/p623_paper_experiments/paper_resource_dataset'),
    Path('../../../gemma4-agent/results/p623_paper_experiments/paper_resource_dataset'),
    Path('.'),
]
DATA = next((p for p in CANDS if (p / 'p670_corrected_n15_ast_and_mssft.json').exists()), ROOT / 'data')
FIGDIR = Path('figures') if Path('figures').is_dir() else DATA / 'figures'
OUTDIR = Path('figures') if Path('figures').is_dir() else Path('.')
print('DATA=', DATA.resolve(), '| FIGDIR=', FIGDIR)

def load(name):
    p = DATA / name
    assert p.exists(), f'{name} missing from companion dataset at {DATA}'
    return json.loads(p.read_text())

# Table 1: recomputed from the 129-task census aggregate + non-holdout ledger + pin unlocks
agg = load('task_soundness_129_aggregate.json')
pins = load('table_1b_harness_pin_flips.json')['table_1b']
nh = pd.read_csv(DATA / 'task_soundness_non_holdout.csv')
nh_usable_csv = int((nh['usable'].astype(str) == 'True').sum())
unlocked = next(r for r in pins if 'flipped_to_unlock' in r)['flipped_to_unlock']
table1 = pd.DataFrame([
    {'Bucket':'Usable','n':agg['usable'],'Rule':agg['definition']},
    {'Bucket':'Unusable','n':agg['unusable'],'Rule':'; '.join(f'{k}={v}' for k, v in agg['unusable_error_top'])},
    {'Bucket':'Usable non-holdout','n':nh_usable_csv + unlocked,
     'Rule':f'{nh_usable_csv} usable in ledger ({len(nh)} non-holdout rows) + {unlocked} pin-unlocked'},
])
assert agg['n_rows'] == agg['usable'] + agg['unusable'] == 129
display(table1)

# Table 1b: pin-recovery rows straight from the JSON
rows1b = []
for r in pins:
    eff = {k: v for k, v in r.items() if k not in ('intervention', 'scope', 'sources', 'source')}
    rows1b.append({'Intervention': r['intervention'], 'Scope': r.get('scope'), 'Effect': json.dumps(eff)})
display(pd.DataFrame(rows1b))
fig = FIGDIR / 'fig1_soundness_and_drift.svg'
if fig.exists():
    display(SVG(filename=str(fig)))
else:
    print('Figure 1 SVG not found at', fig)


DATA= /Volumes/pics/kaggle/swe-horizon-bench/data | FIGDIR= figures
            Bucket   n                                                                                 Rule
            Usable 111 usable iff empty_pass and oracle_pass per soundness_check.py (see harness_verify.md)
          Unusable  18                                     oracle_fail=16; empty_unexpected_pass=1; empty=1
Usable non-holdout  85                           80 usable in ledger (98 non-holdout rows) + 5 pin-unlocked
                                      Intervention              Scope                                                                                                                                                                                        Effect
                pin_deps_starlette_band_docker_p11         129_census                                                                                                              {"after": {"usable": 111, "unusable": 18}, "probe_recovered":

## 4. Prompt & Tool-Schema Ledger

Table 2 is a submission ledger, not a factorial grid (knobs co-vary); `m`=min, `c`=calls, `T`=tools (out=8192, think=4096 unless noted).

### Table 2 — Official LB ledger (anonymous configuration names)

| ref | Score | Configuration | Key knobs |
| --- | ---: | --- | --- |
| `57076074` | **0.18** | **Config-9T-Compact** | 8m, 240s, 9T, ∼4.7 kB |
| `57538500` | **0.18** | **Config-9T-7m** | 7m, out **6144**, 240s, 9T |
| `57703722` | **0.18** | **Config-6T-Temp1** | 8m, temp **1.0**, 240s, 6T |
| `58010294` | **0.18** | **Config-6T-Strict** | 7m, 6144, 240s, 6T |
| `57665916` | **0.13** | **Config-9T-UncalibratedGraph** | +3 unused graph tools (**−0.04**) |
| `57946188` | **0.13** | **HorizonGuard-9T-7m** | 7m, 240s; +338 B prompt |
| `57848086` | **0.12** | **Baseline-9T-8m** | 8m, 240s, ref pack |
| `57773628` | **0.12** | **Config-6T-Timeout300** | 8m, **300s**, 6T |
| `57425395` | **0.17** | **LoRA-v2-r64** | smoke **0.30** |
| `57594583`/`57570012` | **0.12**/**empty** | **LoRA-v3-r64** | smoke **0.30**; 6m / 8m |

Each row is one draw (∼0.017/task; <0.03 gaps are noise). All four **0.18** winners share `timeout_seconds=240` and 28 calls (`57773628` at 300 s dropped to **0.12**); unused graph tools (`57665916`) or +338 B prompt rules (`57946188`) drop LB to **0.13**.

---

## 5. Serving & Horizon Diagnosis

### 5.1 Serving mechanism and one throughput framework

At `batch_size=1`, Punica/SGMV adds **∼840 bf16 launches/token** (7 projections × 2 shrink/expand × 60 layers) outside fused 4-bit `CompressedTensorsW4A16`. With no server-side decode rate logged, every tok/s figure is **episode throughput**, $R=\sum \text{completion tokens}/\sum \text{episode wall}$ (prefill, tools, sandbox I/O). Rows share metric and adapter (only the adapter flag changes) and differ only in sample, pooling, and 480 s timeouts.

### Table 3 — Episode throughput, base → LoRA

| Sample | Pairs | Aggregation | Base → LoRA tok/s | Δ | Capped arms (B / L) |
| --- | ---: | --- | :---: | ---: | :---: |
| Smoke split | 10 | pooled | 16.59 → 12.16 | −27% | 3 / 7 |
| Timing core (§5.2) | 5 | pooled | **20.09 → 11.01** | **−45%** | 0 / 0 |
| Both cohorts | 15 | per-episode mean | 18.13 → 12.10 | −33% | 3 / 7 |
| Both arms uncensored | 8 | pooled | 19.74 → 11.95 | −39% | 0 / 0 |

Throughput drops 27–45% on every slice; the uncensored timing core is the cleanest estimate. Comparing two capped LoRA runs hides this drop (`13.95→12.10`). LoRA packs at smoke **0.30** scored **0.17 / 0.12 / empty** (Table 2).

### 5.2 Paired base-vs-LoRA: $N=5$ timing + $N=15$ with right-censoring

**Timing core ($N=5$, Figure 2).** Toggling only the adapter keeps Pass@1 at **0.40** (same two tasks) while raising wall **300.9→388.0 s** (+87.2 s), cutting tokens **6043→4272** (−29%) and episode throughput **20.09→11.01 tok/s** (−45%), and pushing 120-task wall **10.0→12.9 h**. Fewer tokens save ∼88 s; slower serving adds ∼175 s.

![Figure 2](figures/fig2_paired_e2e_wall_attribution.svg)

**Expanded $N=15$ (exploratory).** Adding the 10 smoke-split pairs to the timing core gives 15 pairs over **10 unique tasks** (5 repeated); **7/15** hit the **480 s** cap on ≥1 arm (**3** on both).

| Estimand (LoRA − Base) | Mean | 95% CI; exact $p$ |
| --- | ---: | --- |
| $\Delta$ wall s ($N=15$) | **+90.5** | $[+49,+136]$; $p<0.001$ |
| $\Delta$ episode tok/s ($N=15$) | **−6.0** | $[−8.1,−4.1]$; $p<0.001$ |
| $\Delta$ steps ($N=15$ / 10-task cluster) | −5.47 / −5.75 | $[−9.3,−1.4]$; $p=0.024$ / $0.043$ |
| $\Delta$ steps (both arms uncensored, $n=8$) | −3.5 | $[−8.8,+2.3]$; $p=0.28$ |

Pass@1 is Base **4/15** vs LoRA **5/15**: LoRA is slower, not less accurate. Fewer steps are mostly a timeout artifact and vanish on uncensored pairs ($p=0.28$).

In [7]:

import math, random, statistics
from pathlib import Path
import json, pandas as pd, matplotlib.pyplot as plt

# SGMV architectural accounting
proj, sgmv_pair, layers = 7, 2, 60
launches = proj * sgmv_pair * layers
print(f'SGMV extra launches/token ≈ {launches} (7×2×60)')

p670 = load('p670_corrected_n15_ast_and_mssft.json')  # MS-SFT tables only; its N=15 pairing (B1 LoRA as base) is retracted
p708 = load('p708_throughput_framework_and_true_base_n15.json')
pairs = p708['corrected_n15']['pairs']

# Table 3: one metric (episode throughput = completion tokens / episode wall), recomputed from raw pairs
import itertools
from collections import defaultdict
def rate_rows(eps):
    return {'n': len(eps), 'pooled': sum(e['tokens'] for e in eps) / sum(e['wall_s'] for e in eps),
            'per_episode': statistics.mean(e['tokens'] / e['wall_s'] for e in eps),
            'capped_480': sum(e['wall_s'] >= 480 for e in eps)}
smoke = [p for p in pairs if p['cohort'] == 'p612_A_vs_B2']; core = [p for p in pairs if p['cohort'] == 'p623_A_vs_B']
unc = [p for p in pairs if p['base']['wall_s'] < 480 and p['lora']['wall_s'] < 480]
t3 = []
for name, ps, agg in [('Smoke split', smoke, 'pooled'), ('Timing core', core, 'pooled'),
                      ('Both cohorts', pairs, 'per_episode'), ('Both arms uncensored', unc, 'pooled')]:
    b, l = rate_rows([p['base'] for p in ps]), rate_rows([p['lora'] for p in ps])
    t3.append({'sample': name, 'pairs': len(ps), 'aggregation': agg, 'base_tok_s': b[agg], 'lora_tok_s': l[agg],
               'delta_pct': 100 * (l[agg] - b[agg]) / b[agg], 'capped_base': b['capped_480'], 'capped_lora': l['capped_480']})
display(pd.DataFrame(t3).round(2))
print('No server-side decode rate was logged; every tok/s above includes prefill, tool and sandbox I/O time.')

def exact_wilcoxon(deltas):
    dz = [d for d in deltas if d != 0]
    pos = defaultdict(list)
    for i, v in enumerate(sorted(abs(d) for d in dz), 1):
        pos[v].append(i)
    rank = {v: statistics.mean(ix) for v, ix in pos.items()}
    rk = [rank[abs(d)] for d in dz]
    w_plus = sum(r for d, r in zip(dz, rk) if d > 0)
    W = min(w_plus, sum(rk) - w_plus)
    cnt = sum(1 for signs in itertools.product([0, 1], repeat=len(dz))
              if min(sum(r for s, r in zip(signs, rk) if s), sum(rk) - sum(r for s, r in zip(signs, rk) if s)) <= W + 1e-9)
    return {'n_nonzero': len(dz), 'W': W, 'p_exact_two_sided': cnt / 2 ** len(dz)}

def boot_ci(xs, n_boot=10000, seed=0):
    rnd = random.Random(seed); n = len(xs)
    ms = sorted(statistics.mean(rnd.choice(xs) for _ in range(n)) for _ in range(n_boot))
    return ms[int(0.025 * n_boot)], ms[int(0.975 * n_boot)]

d = lambda p, k: p['lora'][k] - p['base'][k]
d_wall = [d(p, 'wall_s') for p in pairs]
d_rate = [p['lora']['tokens'] / p['lora']['wall_s'] - p['base']['tokens'] / p['base']['wall_s'] for p in pairs]
d_all = [d(p, 'steps') for p in pairs]
d_unc = [d(p, 'steps') for p in unc]
by_task = defaultdict(list)
for p in pairs:
    by_task[p['task_id']].append(d(p, 'steps'))
d_task = [statistics.mean(v) for v in by_task.values()]
print(f'N={len(pairs)} base-vs-LoRA pairs over {len(by_task)} unique tasks (same p68_v3 adapter in both cohorts)')
for lab, xs in [('Δwall s', d_wall), ('Δepisode tok/s', d_rate), ('Δsteps N=15', d_all),
                ('Δsteps 10-task cluster', d_task), ('Δsteps both-arms-uncensored', d_unc)]:
    print(f'{lab:28s} n={len(xs):2d} mean={statistics.mean(xs):+.2f} CI95={tuple(round(x, 2) for x in boot_ci(xs))}', exact_wilcoxon(xs))
print('Pass@1 base', sum(p['base']['resolved'] for p in pairs), '/15  lora', sum(p['lora']['resolved'] for p in pairs), '/15')

rows = [{'task': p['task_id'], 'cohort': p['cohort'], 'base_wall': p['base']['wall_s'], 'lora_wall': p['lora']['wall_s'],
         'base_tokens': p['base']['tokens'], 'lora_tokens': p['lora']['tokens'],
         'base_steps': p['base']['steps'], 'lora_steps': p['lora']['steps'],
         'base_pass': p['base']['resolved'], 'lora_pass': p['lora']['resolved']} for p in pairs]
display(pd.DataFrame(rows))
fig = FIGDIR / 'fig2_paired_e2e_wall_attribution.svg'
if fig.exists():
    from IPython.display import SVG, display as dsvg
    dsvg(SVG(filename=str(fig)))


SGMV extra launches/token ≈ 840 (7×2×60)
              sample  pairs aggregation  base_tok_s  lora_tok_s  delta_pct  capped_base  capped_lora
         Smoke split     10      pooled       16.59       12.16     -26.70            3            7
         Timing core      5      pooled       20.09       11.01     -45.19            0            0
        Both cohorts     15 per_episode       18.13       12.10     -33.25            3            7
Both arms uncensored      8      pooled       19.74       11.95     -39.43            0            0
No server-side decode rate was logged; every tok/s above includes prefill, tool and sandbox I/O time.
N=15 base-vs-LoRA pairs over 10 unique tasks (same p68_v3 adapter in both cohorts)
Δwall s                      n=15 mean=+90.53 CI95=(48.55, 136.84) {'n_nonzero': 15, 'W': 5, 'p_exact_two_sided': 0.0006103515625}
Δepisode tok/s               n=15 mean=-6.03 CI95=(-7.98, -4.06) {'n_nonzero': 15, 'W': 2, 'p_exact_two_sided': 0.00018310546875}
Δsteps N

### 5.3 Post-localization failure analysis

#### 5.3.1 Post-localization drop and the read→edit gap

We track two nested cohorts: **85** usable non-holdout tasks and the **44**-task smoke+dev subset. On $N=44$, gold read **84.1%** drops to gold edit **40.9%**, post-edit pytest **18.2%**, and pass **15.9%** (Figure 3): **19/30** read-gold failures never edit gold, and **11** edit then fail (missing companion hunk **6/11**, scratch in diff **3/11**). Across **138** post-edit pytest runs, `edit_1` passes **43/138 (31.2%)**; when `edit_1` fails (**52**), only **10** try a second edit+pytest and **0/10** recover (**first-shot edit dominance**). Passing runs edit all companions before pytest more often (**75.2%** vs **56.8%**, $n=27$).

![Figure 3](figures/fig3_post_localization_failure_taxonomy.svg)

#### 5.3.2 Oracle topology and the 150-line horizon law

`read_file` caps at **150 lines / 10k chars** (Figure 4). Across $n=85$ code diffs (**37** single-hunk, **19** single-file multi-hunk, **27** multi-file, **2** metadata-only), Pass@1 falls **27.8% → 11.1% → 5.6%**: single-hunk drops **50%→10%** past line 150; multi-hunk **>150 lines apart** hits **0%** edit/pass; multi-file all-gold edit is **5.6%** despite **89%** shared symbols.

![Figure 4](figures/fig4_topology_horizon_collapse.svg)

#### 5.3.3 Hunk-budget feasibility and the minimal-sufficient paradox

By hunk count ($n=83$; Figure 5), Pass@1 is **25.9%** on 1–3 hunks (**54** tasks) vs **0%** on 4–8 (**16**) and ≥9 (**13**). Across 29 solved tasks, **22/29** winners are **minimal-sufficient** (fewer hunks than the human PR), and **`edit_file` (27/29)**—never `write_file` (**0/29** on gold; **20/29** scratch)—makes the fix.

![Figure 5](figures/fig5_hunk_feasibility_and_n3_taxonomy.svg)

#### 5.3.4 Usable non-holdout hierarchy ($n=85$) and native verification

Across all runs, **44/85 (51.8%)** tasks are solved at least once. In **76** failed episodes (**15** scratch-only, **61** package-edit: **17** assertion, **22** exception, **18** collection, **4** timeout), **14/19** right-function edits on never-solved tasks still fail (**11/14** on primary assertions), motivating quick `python3 -c` checks.

---

In [9]:

# Funnel: pool the two historical cohorts (smoke N=10 + dev N=34 = N=44) from stage counts; timing core N=5 alongside
p627 = load('p627_localization_to_pass_funnel.json')
coh = {c['label']: c for c in p627['cohorts']}
hist = [coh['historical_smoke_base_N10'], coh['historical_dev_base_N35']]
n44 = sum(c['n'] for c in hist)
def pooled(stage):
    return sum(c[f'{stage}_count'] for c in hist) / n44
core = coh['p623_paired_core5_armA_base']
funnel = pd.DataFrame([
    {'stage': lbl, f'hist_N{n44}': round(pooled(st), 3), 'timing_N5': core[f'{st}_pct'] / 100}
    for st, lbl in [('stage1', 'gold_read'), ('stage2', 'gold_edit'), ('stage3', 'post_edit_pytest'), ('stage4', 'pass')]
])
print('funnel definitions:', p627['definition'])

# Topology: functional census counts (p645) × historical pass by topology (p640)
p645 = load('p645_functional_code_topology_and_p624b_exact_diff.json')
p640 = load('p640_oracle_patch_topology_vs_pass.json')
fc = p645['census_n85']['functional_topology_counts']
fb = p640['agent_funnel_by_topology_hist_smoke_dev']
TOPO_NAMES = {
    'S1_single_file_single_hunk': 'Single-hunk (1 file)',
    'S2_single_file_multi_hunk': 'Multi-hunk (1 file)',
    'S3_multi_file': 'Multi-file',
}
topo = pd.DataFrame([
    {'topology': TOPO_NAMES[k], 'census_n85': fc[k], 'hist_n': fb[k]['n'], 'hist_pass': fb[k]['stage4_pass_pct'] / 100}
    for k in ['S1_single_file_single_hunk', 'S2_single_file_multi_hunk', 'S3_multi_file']
])
print('Multi-hunk split by hunk span (≤150 vs >150 lines):', {k: (v['n'], v['stage4_pass_pct']) for k, v in p640['agent_funnel_S2_split_by_delta_L'].items()})

# Hunk tiers: census tier counts × historical pass by tier (p646)
p646 = load('p646_hunk_budget_feasibility_audit.json')
tc = p646['census_n85']['tier_counts']; tp = p646['hist_smoke_dev_pass_by_tier']
hunk = pd.DataFrame([
    {'hunks': p646['tier_definition'][k.split('_')[0]], 'census_n85': tc[k], 'hist_n': tp[k]['n'], 'hist_pass': tp[k]['pass_pct'] / 100}
    for k in ['F1_1to3', 'F2_4to8', 'F3_ge9']
])
display(funnel); display(topo); display(hunk)
from IPython.display import SVG, display as dsvg
for name in ['fig3_post_localization_failure_taxonomy.svg','fig4_topology_horizon_collapse.svg','fig5_hunk_feasibility_and_n3_taxonomy.svg']:
    p = FIGDIR / name
    if p.exists():
        dsvg(SVG(filename=str(p)))


funnel definitions: {'stage1': 'read_file hit ≥1 LocBench gold target file', 'stage2': 'edit_file/write_file hit ≥1 LocBench gold target file', 'stage3': 'pytest run_command after ≥1 edit (any file)', 'stage4': 'resolved=True from paired summary CSV for that run', 'heavy_think': 'agent step completion_tokens ≥ 0.85*4096 after gold read, before gold edit'}
Multi-hunk split by hunk span (≤150 vs >150 lines): {'gt_150': (3, 0.0), 'le_150': (6, 16.7)}
           stage  hist_N44  timing_N5
       gold_read     0.841        1.0
       gold_edit     0.409        0.8
post_edit_pytest     0.182        0.2
            pass     0.159        0.4
            topology  census_n85  hist_n  hist_pass
Single-hunk (1 file)          37      18      0.278
 Multi-hunk (1 file)          19       9      0.111
          Multi-file          27      18      0.056
                                                                             hunks  census_n85  hist_n  hist_pass
                                    

## 6. Part II: Methodology — Structural Retrieval & MS-SFT

*Can structural retrieval and efficient trajectory selection address these failures?*

### 6.1 AST-HorizonGraph against strong retrieval baselines

**AST-HorizonGraph** builds a symbol graph $G=(V,E_{\mathrm{import}}\cup E_{\mathrm{call}}\cup E_{\mathrm{sibling}}\cup E_{\mathrm{attr}})$ from issue tokens weighted by sub-token IDF, reading seed headers, distant companions, or coupled files. Nine readers share one budget ($K$ disjoint ≤150-line windows, issue text only) on 46 multi-hunk/multi-file tasks, scored by fraction of pre-patch gold-hunk lines covered. **Gold files** offers only gold files with a gold seed; **full repo** offers all non-test sources without a seed.

| Reader | Gold files @3 / @5 | Full repo @3 / @5 | Repo file recall @3 |
| --- | :---: | :---: | :---: |
| Top-150 (file headers) | 0.36 / 0.37 | 0.11 / 0.15 | 0.38 |
| Sequential pages | 0.39 / 0.45 | 0.10 / 0.13 | 0.17 |
| BM25 windows [19] | **0.55** / 0.65 | 0.19 / 0.28 | 0.40 |
| Dense, AST chunks (Jina-code [20]) | 0.48 / 0.59 | **0.27 / 0.36** | **0.51** |
| AST-only (def chunks + BM25) | 0.45 / 0.52 | 0.19 / 0.24 | 0.38 |
| Call-graph expansion (2-hop) | 0.42 / 0.56 | 0.13 / 0.17 | 0.34 |
| AST-HorizonGraph, uniform weights | **0.55** / 0.67 | 0.11 / 0.15 | 0.21 |
| **AST-HorizonGraph** | 0.54 / **0.69** | 0.13 / 0.18 | 0.22 |
| Oracle window placement | 0.87 / 0.92 | 0.87 / 0.92 | 0.84 |

Given gold files, AST-HorizonGraph beats Sequential (+0.15 at $K=3$, $p<0.001$) and call-graph expansion (+0.12, $p=0.011$) but ties BM25 (−0.003, CI $[−0.10,+0.08]$); setting IDF weights to 1 changes nothing (−0.007). The gain comes from jumping to lexically matching windows, not graph edges or IDF. Across the full repo, it loses to dense retrieval (−0.14, CI $[−0.25,−0.03]$, $p=0.018$), which opens 51% vs 22% of gold files, while oracle placement reaches ≥0.87 (exact on 34/46): the bottleneck is *finding the right file*, not window placement. `ASTHorizonGraph.localize` also branches on pool size, leaking multi-file status under gold-only pools (sign-flip $p$; 10,000-bootstrap CIs).

### 6.2 MS-SFT: serve-cost-aware trajectory *selection* for SFT data

**What MS-SFT is.** A data-curation rule, not a training loss: we score passing rollouts with $\mathcal{J}_{\mathrm{MS}}$ and keep the top one per solved task for unweighted SFT; it sets no token weights and never updates model parameters.

$$
\mathcal{J}_{\mathrm{MS}}(\tau)=\mathbb{I}[\mathrm{SoundPass}(\tau)]\cdot\exp\!\Big(-\alpha\tfrac{\mathrm{Tokens}(\tau)}{T_{\max}}-\beta\tfrac{\mathrm{Steps}(\tau)}{N_{\max}}-\gamma\tfrac{\mathrm{Hunks}(\tau)}{\mathrm{Hunks}_{\mathrm{gold}}}-\delta\,\mathbb{I}[\mathrm{Scratch}](\tau)\Big),\qquad \tau^*_t=\arg\max_{\tau\in R_t}\mathcal{J}_{\mathrm{MS}}(\tau)
$$

with $R_t$ the rollouts for task $t$ and hand-set $(\alpha,\beta,\gamma,\delta)=(1.5,1.0,0.5,2.0)$, $T_{\max}=8192$, $N_{\max}=80$. Keeping one winner per task across **43** solved tasks (1,237 non-holdout rollouts) stops easy tasks from dominating SFT.

| Selected corpus (1 traj./task, 43 tasks) | Steps | Tokens | Hunks | Proj. LoRA h/120 | @2048 |
| --- | ---: | ---: | ---: | ---: | ---: |
| Uniform random winner | 28.6 | 4658 | 3.25 | **14.10** | 10.15 |
| Closest to full gold PR | 30.4 | 4970 | 2.23 | **15.05** | 10.83 |
| Fewest steps | 22.6 | 4195 | 2.77 | 12.70 | 9.15 |
| **MS-SFT** ($\tau^*_t$) | **24.1** | **3918** | **2.05** | **11.86** | **8.54** |

**What this table shows, and does not.** Rows measure *selected training data*, not a trained model; wall columns project 120-task hours at 11.01 tok/s under standard (`@4096`) and halved (`@2048`) thinking. While $\mathcal{J}_{\mathrm{MS}}$ rewards shorter patches, compact passing rollouts exist for all 43 solved tasks—cutting tokens **16%** vs random winners and **7%** vs fewest-steps (<12 h projected wall). No MS-SFT adapter has been trained or evaluated (§9).

### 6.3 Synthetic OOD probe and toolkit

We release **`swe-horizon-bench`** (`ASTHorizonGraph`, `ms_sft_selector`, `soundness_gate`, CLI) and **SWE-HorizonBench-OOD**: 60 synthetic targets across eight external libraries (0% ID overlap) appending `# HORIZON_BENCH_FIX` to real `def` lines, checked by marker presence (**11/11** two-sided). Because prompts name target symbols, we treat it as a **construct-validity warning**: symbol-named probes inflate graph recall just as gold-only pools do (§6.1).

---

In [11]:

import sys
from pathlib import Path
# swe_horizon_bench is vendored in the companion dataset (DATA/swe_horizon_bench) and in the repo under src/
for p in [DATA, Path('..') / 'src', Path('../src'), Path('../../swe-horizon-bench/src')]:
    if (p / 'swe_horizon_bench').is_dir():
        sys.path.insert(0, str(p.resolve()))
        break
else:
    raise ImportError('swe_horizon_bench package not found next to the companion data')

from swe_horizon_bench import localize_windows, summarize_policies, classify_topology, two_sided_sound, run_kbench_suite, load_ood_dataframe
from swe_horizon_bench.ast_horizon_graph import ASTHorizonGraph
import tempfile, shutil, textwrap, json, pandas as pd, matplotlib.pyplot as plt

# Live localize demo on a tiny synthetic package
demo = Path(tempfile.mkdtemp(prefix='hz_demo_'))
(demo/'pkg').mkdir()
(demo/'pkg'/'__init__.py').write_text('from .core import DeepHelper\n')
(demo/'pkg'/'core.py').write_text('\n'.join(['def shallow():\n    return 1\n'] + ['# pad']*200 + [
    'class DeepHelper:\n    def compute_token_overlap(self, x):\n        return x\n'
]))
wins = localize_windows(demo/'pkg', 'DeepHelper compute_token_overlap fails', seed_file='core.py', k=3)
print('LIVE localize windows:', wins)
shutil.rmtree(demo, ignore_errors=True)

# Load OOD eval. NOTE: SWE-HorizonBench-OOD is a SYNTHETIC structural probe (paper §6.3): each gold
# patch appends '# HORIZON_BENCH_FIX' to real def lines, the problem statement names the target symbols,
# and 'two-sided verification' checks marker presence. It measures localization geometry, not bug fixing.
ood = load('swe_horizon_bench_ood_eval.json')
ae = ood['ast_eval']
ood_rows = [json.loads(l) for l in (DATA / 'swe_horizon_bench_ood_60.jsonl').read_text().splitlines() if l.strip()]
n_marker = sum('HORIZON_BENCH_FIX' in r['gold_patch'] for r in ood_rows)
print(f'OOD targets: {len(ood_rows)}; gold patches that are marker-comment insertions: {n_marker}/{len(ood_rows)} (synthetic by design)')
print('example problem statement:', ood_rows[0]['problem_statement'][:120])

def row(name, d):
    return {
        'group': name,
        'Top-150@1': d['b1_recall@1']['mean'], 'Graph@1': d['ast_recall@1']['mean'],
        'Seq@2': d['seq_recall@2']['mean'], 'Graph@2': d['ast_recall@2']['mean'],
        'Seq@3': d['seq_recall@3']['mean'], 'Graph@3': d['ast_recall@3']['mean'],
    }
ood_tbl = pd.DataFrame([
    row('Single-hunk ≤150', ae['S1_le150']),
    row('Single-hunk >150', ae['S1_gt150']),
    row('Multi-hunk (1 file, all)', ae['S2']),
    row('Multi-hunk (>150 apart)', ae['S2_deltaL_gt150']),
    row('Multi-file', ae['S3']),
])
print('OOD libs', ood.get('libs')); print('two_sided', ood.get('two_sided_verify'))
display(ood_tbl)

# Paper §6.1: nine readers, one budget (K disjoint <=150-line windows), gold-file vs full-repo pools
p707 = load('p707_horizongraph_baseline_suite.json')
READERS = [('top150','Top-150 (file headers)'), ('sequential','Sequential pages'), ('bm25','BM25 windows'),
           ('embed','Dense, AST chunks (Jina-code)'), ('ast_only','AST-only (def chunks + BM25)'),
           ('callgraph','Call-graph expansion (2-hop)'), ('hg_noidf','AST-HorizonGraph, uniform weights'),
           ('hg','AST-HorizonGraph'), ('oracle','Oracle window placement')]
g7, r7 = p707['summary']['gold']['all'], p707['summary']['repo']['all']
base_tbl = pd.DataFrame([{'reader': lab, 'gold@3': g7[f'{k}@3'], 'gold@5': g7[f'{k}@5'],
                          'repo@3': r7[f'{k}@3'], 'repo@5': r7[f'{k}@5'], 'repo_file@3': r7[f'{k}_file@3']}
                         for k, lab in READERS]).round(2)
print(f"n={p707['n_tasks']} tasks; holdout_leaks={p707['holdout_leaks']}")
display(base_tbl)
pt = pd.DataFrame([{'pool': pool, 'contrast': k, **{kk: v[kk] for kk in ('mean', 'signflip_p', 'wins', 'losses')},
                    'ci_lo': v['ci95'][0], 'ci_hi': v['ci95'][1]}
                   for pool in ('gold', 'repo') for k, v in p707['paired_hg_vs_baselines'][pool].items() if k.endswith('@3')]).round(3)
display(pt)

# MS-SFT policy table from the shipped 1,237-rollout analysis (task-balanced macro over 43 solved tasks)
ms = p670['ms_sft']
print('MS-SFT corpus:', {k: ms[k] for k in ('n_meta_lines', 'n_resolved_non_holdout', 'n_tasks_solved')})
pol = {k: v for k, v in ms.items() if k.startswith('policy') and isinstance(v, dict)}
ms_tbl = pd.DataFrame([
    {'policy': v.get('label', k), 'steps': v.get('mean_steps'), 'tokens': v.get('mean_completion_tokens'),
     'hunks': v.get('mean_hunks'),
     'proj_h120_lora_think4096': v.get('proj_h120_lora_11p01_think4096'),
     'proj_h120_lora_think2048': v.get('proj_h120_lora_11p01_think2048')}
    for k, v in pol.items()
]).round(2)
display(ms_tbl)
print('NOTE: rows are statistics of the per-task argmax-selected SFT corpus (one trajectory per task), not of a trained model; h/120 columns are linear projections at measured LoRA tok/s.')

# MS-SFT toy macro (illustrative API demo)
toy = [
    {'task_id':'a','sound':True,'tokens':5000,'steps':30,'hunks':4,'gold_hunks':2,'scratch':False},
    {'task_id':'a','sound':True,'tokens':2000,'steps':12,'hunks':1,'gold_hunks':2,'scratch':False},
    {'task_id':'b','sound':True,'tokens':4000,'steps':25,'hunks':3,'gold_hunks':2,'scratch':True},
    {'task_id':'b','sound':True,'tokens':2200,'steps':14,'hunks':1,'gold_hunks':2,'scratch':False},
]
print('MS-SFT policies', summarize_policies(toy))

# Native @kbench.task suite (offline shim or live kaggle_benchmarks SDK)
ood_df = load_ood_dataframe(DATA / 'swe_horizon_bench_ood_60.jsonl')
kb_board = run_kbench_suite(df=ood_df)
print('@kbench.task suite (kaggle.com/benchmarks-compatible):')
display(kb_board if hasattr(kb_board, 'head') else pd.DataFrame(kb_board))

# 4-panel summary matching Figures 2–5:
# (a) Retrieval across 150-line horizons (in-dist n=46); (b) Localization->edit->verify->resolve funnel (N=44);
# (c) Synthetic probe recall vs in-dist recall vs actual behavioral Pass@1; (d) LoRA serve-cost wall vs 12 h cutoff
p674 = load('p674_upgraded_ast_horizongraph.json')['indomain']
h120 = {r['policy']: r['proj_h120_lora_think4096'] for _, r in ms_tbl.iterrows()}
h_keys = [k for k in h120 if 'uniform' in k.lower()] + [k for k in h120 if 'gold' in k.lower()] + [k for k in h120 if 'MS-SFT' in k]
fig, axs = plt.subplots(2, 2, figsize=(10, 7.5))

# Panel (0,0): nine readers vs K (full-repo pool, p707) — dense/BM25 vs AST-HorizonGraph vs oracle
ks = [1, 2, 3, 5]
for k_, lab_, sty_, col_ in [('hg','AST-HorizonGraph','o-','#16a34a'), ('embed','Dense (Jina-code)','d-','#7c3aed'),
                             ('bm25','BM25 windows','s--','#2563eb'), ('callgraph','Call-graph','v:','#0d9488'),
                             ('sequential','Sequential','^:','#64748b'), ('oracle','Oracle','k-','#0f172a')]:
    axs[0,0].plot(ks, [r7[f'{k_}@{k}'] for k in ks], sty_, color=col_, label=lab_)
axs[0,0].plot(ks, [g7[f'hg@{k}'] for k in ks], 'o--', color='#16a34a', alpha=0.45, label='AST-HG (gold files given)')
axs[0,0].set_xticks(ks); axs[0,0].set_xlabel('150-line windows (K)'); axs[0,0].set_ylim(0, 1.3)
axs[0,0].set_title('Gold-line recall vs K, full repo (n=46)'); axs[0,0].legend(fontsize=6, ncol=2, loc='upper left')

# Panel (0,1): Localization -> edit -> verification -> resolution funnel (N=44)
fn_col = [c for c in funnel.columns if c.startswith('hist_')][0]
axs[0,1].barh(funnel['stage'][::-1], funnel[fn_col][::-1], color=['#16a34a','#0d9488','#2563eb','#2563eb'])
axs[0,1].set_xlim(0, 1.0); axs[0,1].set_title('Funnel: read 84% → edit 41% → pytest 18% → pass 16%')

# Panel (1,0): Synthetic structural recall@3 vs in-dist recall@3 vs behavioral Pass@1
groups = ['1-hunk', 'Multi-hunk', '>150 apart', 'Multi-file']
synth_r3 = [0.94, float(ood_tbl.loc[2,'Graph@3']), float(ood_tbl.loc[3,'Graph@3']), float(ood_tbl.loc[4,'Graph@3'])]
sm7 = p707['summary']['repo']
indist_r3 = [float('nan'), sm7['multi_hunk']['hg@3'], sm7['multi_hunk_gt150']['hg@3'], sm7['multi_file']['hg@3']]
behav_p1 = [float(topo.loc[0,'hist_pass']), float(topo.loc[1,'hist_pass']), 0.0, float(topo.loc[2,'hist_pass'])]
x = range(len(groups)); w = 0.26
axs[1,0].bar([i - w for i in x], synth_r3, width=w, color='#93c5fd', label='Synthetic probe Recall@3')
axs[1,0].bar(list(x), indist_r3, width=w, color='#2563eb', label='Real tasks, full repo, AST-HG Recall@3')
axs[1,0].bar([i + w for i in x], behav_p1, width=w, color='#dc2626', label='Behavioral Pass@1')
axs[1,0].set_xticks(list(x)); axs[1,0].set_xticklabels(groups); axs[1,0].set_ylim(0, 1.05)
axs[1,0].set_title('Structural recall vs behavioral Pass@1'); axs[1,0].legend(fontsize=7)

# Panel (1,1): Causal serve-cost wall vs 12 h cutoff
labels_w = ['Base\n(20.1 tok/s)', 'LoRA N=5\n(11.0 tok/s)'] + ['Uniform', 'Full-gold', 'MS-SFT'][:len(h_keys)]
vals_w = [10.0, 12.9] + [h120[k] for k in h_keys]
cols_w = ['#2563eb', '#c2410c', '#bab0ac', '#e45756', '#16a34a'][:len(vals_w)]
axs[1,1].bar(labels_w, vals_w, color=cols_w)
axs[1,1].axhline(12.0, color='k', ls=':', label='12h host wall')
axs[1,1].set_title('LoRA serve-cost wall vs 12h cutoff (h/120)'); axs[1,1].legend(fontsize=8)
plt.tight_layout()
out_png = OUTDIR / 'fig_p672_four_panel_summary.png'
plt.savefig(out_png, dpi=140)
plt.show()
print('Saved', out_png)


LIVE localize windows: [{'path': 'core.py', 'start_line': 1, 'end_line': 150, 'score': 1.45, 'reason': 'header-bridge'}, {'path': 'core.py', 'start_line': 151, 'end_line': 207, 'score': 5.485639156829384, 'reason': 'deep-page2'}, {'path': '__init__.py', 'start_line': 1, 'end_line': 2, 'score': 2.819205181129452, 'reason': 'DeepHelper'}]
OOD targets: 60; gold patches that are marker-comment insertions: 60/60 (synthetic by design)
example problem statement: [click] Bug: _make_text_stream. The symbols _make_text_stream behave incorrectly; please apply a minimal fix near these 
OOD libs ['anyio', 'click', 'httpcore', 'packaging', 'pluggy', 'starlette', 'typer', 'urllib3']
two_sided {'n': 11, 'n_usable_two_sided': 11, 'rows': [{'task_id': 'ood_click_s1__make_text_stream_22', 'repo': 'click', 'topology': 'S1_single_hunk', 'empty_pass': False, 'oracle_pass': True, 'usable': True, 'sound_rule': 'empty_pass==False AND oracle_pass==True'}, {'task_id': 'ood_typer_s1__make_text_stream_23', 'repo':

## 7. Part III: Validation — Claim Audit & Limitations

*Do these methods improve real software engineering task completion?*

| Claim | Verdict & evidence |
| --- | --- |
| Two-sided gate required; ImportErrors = host-env gaps | **Supported** (**111/129** usable) / **False** (**24/24** = exports/syntax; ceiling **44/85**) |
| LoRA smoke (**0.30**) transfers to LB; LoRA costs no wall time | **Both False** (LB **0.17/0.12/empty**; $N=15$ $\Delta$wall **+90.5 s**, $p<0.001$) |
| LoRA lowers episode throughput; LoRA takes fewer steps | **Supported** (−33%, $p<0.001$) / **Confounded** by censoring (uncensored $p=0.28$) |
| Localization ≫ resolution; 150-line cap hides companions; `edit_file` wins | **Supported** (**84.1%→15.9%**; >150-apart **0%**; `edit_file` **27/29** vs `write_file` **0/29**) |
| AST-HorizonGraph gains come from graph structure | **False** (ties BM25 on gold files; trails dense retrieval at repo scale, $p=0.018$) |
| Full-gold imitation is serve-cost optimal; unused tools/rules are free | **Both False** (worst *projected* **15.05 h**; **−0.05** LB tax in `57665916`/`57946188`; `240s` in **0.18** winners) |

**Limitations.** Single-draw LB rows cannot resolve <0.03 gaps; $N=5$ timed / $N=15$ censored samples are small; no server-side decode rate was logged; the 8-library probe names target symbols and checks markers; and MS-SFT is selection-only, with no trained adapter evaluated.

**Reproducibility.** Dataset `moranzavdi/gemma4-soundness-and-punica-bench` ships all tables, figures, `swe_horizon_bench/`, and the executed notebook.

## 8. Related Work

**Repository-level code retrieval.** Iterative retrieve-and-edit pipelines (RepoCoder [1]) and process benchmarks (*ContextBench* [2]) show line-level sufficiency governs patch success. Our funnel (**84.1%** gold-file read vs **15.9%** pass) isolates where file retrieval succeeds yet a **150-line** `read_file` window truncates companion sites.

**Graph-based code representations.** Program graphs (AST/CFG/PDG; code2vec [3]) and repository symbol graphs (AutoCodeRover [4], RepoGraph [5]) guide localization over unrestricted contexts. **AST-HorizonGraph** adapts this to a 150-line budget, but against BM25 [19], dense [20], AST-only, and call-graph baselines its edges add no recall (§6.1). Conversely, Hu et al. [6] show uncalibrated structure tools raise cost up to **+28.14%**, matching our **−0.05** LB tax (`57665916`).

**Long-context SWE agents.** SWE-agent [7] and Agentless [8] assume generous budgets and frontier APIs. Under a 4-bit host with a 28-call cap and **0/10** second-edit repair, companion-complete *first-shot* edits outweigh multi-turn repair loops.

**Efficient inference for PEFT.** Multi-tenant LoRA engines—Punica/SGMV [9] and S-LoRA [10] atop PagedAttention/vLLM [11]—optimize batched adapters, while DoRA [12] decomposes weight updates. At `batch_size=1` on a 4-bit base, unmerged rank-64 LoRA leaves the fused path (§5.1), cutting episode throughput **45%** and breaching the 12 h wall.

**Trajectory selection and outcome-based SFT.** Rejection-sampled self-training (STaR [13], ReST$^{EM}$ [14]) filters by binary pass, over-weighting easy tasks and keeping verbose trajectories that time out under LoRA; **MS-SFT** replaces binary filtering with a per-task cost-aware argmax before plain SFT.

**Benchmark contamination and soundness.** Following SWE-bench [15], LiveCodeBench [16], SWE-Bench+ [17], and test-overfitting audits [18], our two-sided gate quarantines **18/129** broken tasks and catches empty-green false positives.

---

## 9. Capability Addendum (P7.10–P7.11, MEASURED)

On the 31B QAT host, folded rank-**8** LoRA (`use_dora=false`) restored **~5.7 h**/120 (vs rank-64 Punica **14.2 h** timeout). On-policy GapDPO v1 hit **6/13** (`fork_pref=0.875`); synthetic-$P_{\min}^*$ splices (v2 **4/13**, v1.5 **1/9**) raised logit drift and broke T0 guards. Fresh-KV Clean-Critic flipped `rich_3052` (**projected 7/13**). E4B GapDPO **2/13**; pure MS-SFT+3240B lean prompt (**E4B-D21**) is staged for the $10k$ track.

---

## 10. Conclusion

Under a 12 h budget, score is set by soundness (**111/129**), serving speed ($N=15$ censored **−33%** tok/s), retrieval (**84%→16%**; 150-line cap), and first-shot edits (**0/10** repair)—not weights alone. Practical playbook: base/`timeout=240`, compact 6–9T, **0.18** LB floor, dense retrieval before windowing, **MS-SFT**, and on-policy preference over synthetic-$P_{\min}^*$ grafts.

---

## 11. References

1. F. Zhang et al. (2023). *RepoCoder: Repository-Level Code Completion Through Iterative Retrieval and Generation.* EMNLP, arXiv:2303.12570.
2. H. Li et al. (2026). *ContextBench: A Benchmark for Context Retrieval in Coding Agents.* arXiv:2602.05892.
3. U. Alon et al. (2019). *code2vec: Learning Distributed Representations of Code.* POPL, PACMPL 3:40, arXiv:1803.09473.
4. Y. Zhang et al. (2024). *AutoCodeRover: Autonomous Program Improvement.* ISSTA, arXiv:2404.05427.
5. S. Ouyang et al. (2024). *RepoGraph: Enhancing AI Software Engineering with Repository-level Code Graph.* arXiv:2410.14684.
6. Y. Hu et al. (2026). *Analyzing and Mitigating Cost-Inefficient Behaviors in Coding Agents.* arXiv:2609.30725.
7. J. Yang et al. (2024). *SWE-agent: Agent-Computer Interfaces Enable Automated Software Engineering.* NeurIPS, arXiv:2405.15793.
8. C. S. Xia et al. (2024). *Agentless: Demystifying LLM-based Software Engineering Agents.* arXiv:2407.01489.
9. L. Chen et al. (2023). *Punica: Multi-Tenant LoRA Serving.* MLSys 2024, arXiv:2310.18547.
10. Y. Sheng et al. (2023). *S-LoRA: Serving Thousands of Concurrent LoRA Adapters.* MLSys 2024, arXiv:2311.03285.
11. W. Kwon et al. (2023). *Efficient Memory Management for Large Language Model Serving with PagedAttention.* SOSP, arXiv:2309.06180.
12. S.-Y. Liu et al. (2024). *DoRA: Weight-Decomposed Low-Rank Adaptation.* ICML, arXiv:2402.09353.
13. E. Zelikman et al. (2022). *STaR: Bootstrapping Reasoning With Reasoning.* NeurIPS, arXiv:2203.14465.
14. A. Singh et al. (2023). *Beyond Human Data: Scaling Self-Training for Problem-Solving with Language Models.* TMLR 2024, arXiv:2312.06585.
15. C. E. Jimenez et al. (2024). *SWE-bench: Can Language Models Resolve Real-World GitHub Issues?* ICLR, arXiv:2310.06770.
16. N. Jain et al. (2024). *LiveCodeBench: Holistic and Contamination Free Evaluation of Large Language Models for Code.* ICLR 2025, arXiv:2403.07974.
17. R. Aleithan et al. (2024). *SWE-Bench+: Enhanced Coding Benchmark for LLMs.* arXiv:2410.06992.
18. T. Ahmed et al. (2025). *Investigating Test Overfitting on SWE-bench.* arXiv:2511.16858.
19. S. Robertson and H. Zaragoza (2009). *The Probabilistic Relevance Framework: BM25 and Beyond.* Foundations and Trends in Information Retrieval 3(4):333–389.
20. M. Günther et al. (2023). *Jina Embeddings 2: 8192-Token General-Purpose Text Embeddings for Long Documents.* arXiv:2310.19923.